# COT Forecasting Research

In [59]:
import pandas as pd
import numpy as np
from pathlib import Path
from cot_pipeline.signals import compute_signals

# project root = nearest folder (cwd or a parent) holding config.py, so paths work from docs/ or the root
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.py").exists())
DATA_DIR = PROJECT_ROOT / "data"
MARKET_COL = "Market and Exchange Names"
DATE_COL = "As of Date in Form YYYY-MM-DD"

'''
special fields for analysing the commitment of traders report
'''
SPECIAL_COLUMNS = [
    MARKET_COL, DATE_COL, "Noncommercial Positions-Long (All)",
    "Noncommercial Positions-Short (All)" , "Change in Noncommercial-Long (All)", 
    "Change in Noncommercial-Short (All)", "Open Interest (All)", "Change in Open Interest (All)"
]

In [60]:
gold_df = pd.read_csv(DATA_DIR / "GOLD.csv")[SPECIAL_COLUMNS]
gold_df.shape

(2058, 8)

In [61]:
# add net positions and change in net positions columns to the dataframe
gold_df["net_positions"] = gold_df["Noncommercial Positions-Long (All)"] - gold_df["Noncommercial Positions-Short (All)"]
gold_df["Change Net Positions"] = gold_df["Change in Noncommercial-Long (All)"] - gold_df["Change in Noncommercial-Short (All)"]
gold_df.head()

,Market and Exchange Names,As of Date in Form YYYY-MM-DD,Noncommercial Positions-Long (All),Noncommercial Positions-Short (All),Change in Noncommercial-Long (All),Change in Noncommercial-Short (All),Open Interest (All),Change in Open Interest (All),net_positions,Change Net Positions
0,GOLD - COMMODITY EXCHANGE INC.,2026-10-13,245300,37421,-103.0,2353.0,396345,236.0,207879,-2456.0
1,GOLD - COMMODITY EXCHANGE INC.,2026-10-06,245403,35068,-4333.0,3964.0,396109,-10347.0,210335,-8297.0
2,GOLD - COMMODITY EXCHANGE INC.,2026-09-29,249736,31104,-4246.0,2975.0,406456,-6344.0,218632,-7221.0
3,GOLD - COMMODITY EXCHANGE INC.,2026-09-22,253982,28129,-4077.0,408.0,412800,2901.0,225853,-4485.0
4,GOLD - COMMODITY EXCHANGE INC.,2026-09-15,258059,27721,-2948.0,-1326.0,409899,-1328.0,230338,-1622.0


In [62]:
# compute signals for the gold dataframe
gold_df = compute_signals(gold_df)

In [63]:
#rename special columns to more readable names
gold_df.rename(columns={
    "Market and Exchange Names": "market",
    "As of Date in Form YYYY-MM-DD": "date",
    "Noncommercial Positions-Long (All)": "long",
    "Noncommercial Positions-Short (All)": "short",
    "Change in Noncommercial-Long (All)": "change_long",
    "Change in Noncommercial-Short (All)": "change_short",
    "Open Interest (All)": "open_interest",
    "Change in Open Interest (All)": "change_open_interest",
    "Change Net Positions": "change_net_positions",
}, inplace=True)  

In [64]:
gold_df.head()

,market,date,long,short,change_long,change_short,open_interest,change_open_interest,net_positions,change_net_positions,signal,Interpretation
0,GOLD - COMMODITY EXCHANGE INC.,2026-10-13,245300,37421,-103.0,2353.0,396345,236,207879,-2456.0,2,Bearish
1,GOLD - COMMODITY EXCHANGE INC.,2026-10-06,245403,35068,-4333.0,3964.0,396109,-10347,210335,-8297.0,3,Bearish Reversal
2,GOLD - COMMODITY EXCHANGE INC.,2026-09-29,249736,31104,-4246.0,2975.0,406456,-6344,218632,-7221.0,3,Bearish Reversal
3,GOLD - COMMODITY EXCHANGE INC.,2026-09-22,253982,28129,-4077.0,408.0,412800,2901,225853,-4485.0,2,Bearish
4,GOLD - COMMODITY EXCHANGE INC.,2026-09-15,258059,27721,-2948.0,-1326.0,409899,-1328,230338,-1622.0,3,Bearish Reversal


In [65]:
# add a column to record extreme netposition as well as the extreme reading on a scale of 0-100% 
abs_net_positions = gold_df["net_positions"].abs()

# Percentile score for the magnitude of each net position (0–100)
gold_df["net_position_extremeness_pct"] = abs_net_positions.rank(pct=True) * 100

# Flag readings in the lowest or highest 5% by absolute magnitude
gold_df["extreme_net_position"] = (
    (abs_net_positions <= abs_net_positions.quantile(0.05))
    | (abs_net_positions >= abs_net_positions.quantile(0.95))
)

In [67]:
extreme_gold = gold_df[gold_df["extreme_net_position"] == True]
extreme_gold.head(2)

,market,date,long,short,change_long,change_short,open_interest,change_open_interest,net_positions,change_net_positions,signal,Interpretation,net_position_extremeness_pct,extreme_net_position
55,GOLD - COMMODITY EXCHANGE INC.,2025-09-23,332808,66059,6030.0,5691.0,528789,12568,266749,339.0,1,Bullish,96.307094,True
56,GOLD - COMMODITY EXCHANGE INC.,2025-09-16,326778,60368,1903.0,-2767.0,516221,6596,266410,4670.0,1,Bullish,96.161322,True


## Part 2: Adding the gold price

**What we are doing:** bringing in gold's daily price so we can ask whether speculator positioning tells us
anything about where the price goes next.

**Which price?** We use **COMEX gold futures** (Yahoo ticker `GC=F`). This is the same market the COT report
counts contracts in, and its history starts in August 2000. The gold ETF `GLD` moves almost identically
(their weekly moves are 98% correlated) but only starts in November 2004, so we would lose four years.

**Why read from a file instead of calling Yahoo here?** Yahoo limits how often you can download. The download
runs once in a terminal with `python scripts/fetch_prices.py` and is saved to `data/prices/GOLD_GC=F.csv`;
the notebook just reads that file. To get newer prices later, run
`python scripts/fetch_prices.py --refresh`.

In [ ]:
# pull price data for gold from Yahoo Finance and update the dataframe with the price data
# (downloaded once by scripts/fetch_prices.py; here we read the saved file)
from cot_pipeline import prices as price_data

gold_price_data = price_data.read_saved_prices()
print(f"{len(gold_price_data):,} trading days from {gold_price_data['date'].min():%Y-%m-%d} "
      f"to {gold_price_data['date'].max():%Y-%m-%d}")
gold_price_data.tail()

## Part 3: Unsupervised learning, explained simply

### What is unsupervised learning?
Normally a computer is trained with **right answers** ("this email is spam, this one isn't") and learns to
predict them. That is *supervised* learning.

In **unsupervised** learning there are no right answers. We hand over the data and ask the computer to
**find structure by itself**. The most common kind is **clustering**: sorting things into groups so that
members of a group are alike and different groups are not. Think of sorting a big pile of photos into piles
without anyone telling you what the piles should be.

Here, each "thing" is **one week** of the COT report, and we ask: *do the weeks fall into a few recurring
types?* We call each type a **regime** (a recurring state the market is in).

### The methods we tried

| Method | How it works, in everyday terms | Good at | Weak at |
|---|---|---|---|
| **K-Means** | Drop *k* pins on the data; every week joins its nearest pin; move each pin to the middle of its group; repeat until nothing changes. | Simple, fast, every group has a clear "average week" | Assumes roughly round groups; you choose *k* |
| **Gaussian mixture** | Like K-Means, but each group is a soft oval cloud, and each week gets a *probability* of belonging to each group. | Overlapping groups | Needs more data; can keep splitting |
| **Hierarchical clustering (Ward)** | Start with every week alone; keep merging the two most similar groups, building a family tree; cut the tree into *k* groups. | Shows how groups nest | Can change a lot with small changes in the data |
| **DBSCAN** | Look for crowded areas separated by empty space, like spotting towns on a map at night. | Islands of any shape, flags outliers | Finds one big blob when there are no gaps |
| **PCA** | *Not* a clustering method: squeezes many columns into two or three "summary" columns, mainly to draw the data. | Drawing many columns on a flat chart | Summary columns are hard to interpret |

We don't need PCA here: we can plot two real, easy-to-read columns instead (speculators' longs vs shorts).

### How we judge whether the groups are any good

| Score | Plain meaning | Good value |
|---|---|---|
| **Silhouette** (-1 to 1) | For each week: is it much closer to its own group than to the next one? | Higher. Near 1 = clearly in its group, 0 = on the border |
| **Davies-Bouldin** | How much the groups overlap | Lower |
| **Stability** (0 to 1) | Re-run the method on random 80% slices of history: do the same weeks end up together? | Near 1 = the groups are real, not luck |
| **Average weeks per regime** | How long the market stays in a group before switching | Several weeks: a "state", not noise |
| **BIC** | Gaussian-mixture score that rewards fit but penalises extra groups | Lower; if it never stops falling, there's no natural number of groups |

### How we judge the results
- **Correlation** (-1 to 1): how closely two things move together. 1 = always together, 0 = unrelated,
  -1 = always opposite.
- **p-value**: roughly, *how likely we'd see differences this big if there were really no difference*.
  Below **0.05** (5%) we treat a difference as real; above that it could easily be luck.
- **Kruskal-Wallis test**: a standard check of whether several groups really differ. It compares rankings
  rather than raw values, so a few extreme weeks can't dominate.

### The columns we use

| Column | What it means |
|---|---|
| `long_share` | Speculators' long contracts (bets on a rise) as a share of all open contracts |
| `short_share` | Speculators' short contracts (bets on a fall) as a share of all open contracts |
| `net_share` | Longs minus shorts as a share of open contracts. Positive = net long |
| `cot_index` | Where this week's net position sits in its last-3-years range: 0 = most short, 1 = most long |
| `net_flow_4w` | How much speculators bought (+) or sold (-) over the last 4 weeks, as a share of the market |
| `oi_growth_4w` | % change in open contracts over the last 4 weeks (money coming in or leaving) |
| `close` | Gold's price on the report date (US$ per ounce) |
| `return_13w` | Gold's % change over the last 13 weeks (the recent trend) |
| `price_pct_3y` | Where gold's price sits in its last-3-years range: 0 = lowest, 1 = highest |
| `forward_net_flow` | What speculators did over the **next 8 weeks** (used only to check results, never to build groups) |
| `forward_return` | Gold's % change over the **next 8 weeks**, starting after the report is published (only for checking) |

**Why shares instead of raw contract counts?** The market today is several times bigger than in 1986, so
raw counts mostly measure market size. Shares make 1990 and 2026 comparable.

In [ ]:
import config
from cot_pipeline import regimes
from cot_pipeline import regime_plots as rplots

%matplotlib inline

# gold_df is newest-first and mixes three gold contracts; keep the main COMEX contract, oldest-first
gold_frame = regimes.prepare_market_frame(gold_df)
gold_features = regimes.build_features(gold_frame)
# add gold's price on each report date and what it did before and after
gold_features = price_data.add_price_features(gold_features, gold_price_data)

friendly = {"date": "Report date", "long_share": "Longs (share)", "short_share": "Shorts (share)",
            "net_share": "Net (share)", "cot_index": "3-yr percentile of net", "close": "Gold price ($)",
            "return_13w": "Gold, last 13 wks", "forward_return": "Gold, next 8 wks"}
gold_features[list(friendly)].dropna().tail().rename(columns=friendly).round(3)

### Step 1: why we use shares of the market
**What the chart shows:** top, the total number of open gold contracts since 1986; bottom, speculators' net
position as a share of that total.

**How to read it:** the top line roughly triples, so raw counts would make recent years look extreme just
because the market is bigger. The bottom line is on the same scale throughout, so the model can compare
1995 with 2025 fairly.

In [ ]:
rplots.scale_free_figure(gold_frame, gold_features);

### Step 2: which method finds the best groups?
We run every method from the table above with 2 to 7 groups and score each one. The six panels each answer
one question; the circled point is our choice. *This cell takes about 30 seconds.*

1. **Separation (silhouette):** higher = clearer groups.
2. **Stability:** near 1 = the same groups come back on different slices of history.
3. **Regime length:** how many weeks the market typically stays in a group.
4. **Inputs:** only "where speculators stand" vs also adding the last 4 weeks of buying and selling.
5. **Gaussian mixture BIC:** does it ever settle on a number of groups?
6. **DBSCAN:** does it find separate islands?

In [ ]:
state_columns = config.STATE_FEATURES
flow_columns = config.STATE_FEATURES + config.FLOW_FEATURES
state_matrix = regimes.scale_features(regimes.complete_rows(gold_features, state_columns), state_columns)
flow_matrix = regimes.scale_features(regimes.complete_rows(gold_features, flow_columns), flow_columns)

scorecard = regimes.model_scorecard(state_matrix)
flow_scorecard = regimes.model_scorecard(flow_matrix, clusterers={"K-Means": regimes.kmeans_labels})
bic = regimes.gmm_bic(state_matrix)
dbscan = regimes.dbscan_summary(state_matrix)

rplots.model_selection_figure(scorecard, flow_scorecard, bic, dbscan);

**What we found (at 4 groups):**

| Method | Separation | Stability | Verdict |
|---|---|---|---|
| **K-Means** | **0.37**, best at every number of groups | 0.96 | **Chosen** |
| Gaussian mixture | 0.32 | 0.97 | Close second, but its BIC keeps falling, so it never picks a natural number of groups |
| Hierarchical (Ward) | 0.36 | 0.66 | Groups change a lot when part of history is left out |
| DBSCAN | n/a | n/a | Puts nearly all weeks in one blob: positioning is one continuous cloud, not islands |

- **Why 4 groups?** 2 groups separate slightly better but only split "long-heavy" from "not". 4 groups keep
  almost the same separation, come back reliably, and last about 8 weeks each.
- **Why not add the last 4 weeks of buying and selling?** Panel 4: it makes every grouping blurrier. Weekly
  flows are noisy, and they describe *what just happened* rather than *where speculators stand*.
- Separation of 0.37 is "reasonable" rather than "crisp": the regimes are real but their borders are soft,
  like neighbourhoods in one city rather than separate towns.

### Step 3: the four regimes
K-Means numbers its groups arbitrarily, so we sort them from most short to most long and name them after
what their average week looks like:

| Name | What it means |
|---|---|
| **Net short / 3-yr low** | Speculators hold more shorts than longs, or are at their most bearish in 3 years |
| **Low participation** | Few speculator positions on either side; mostly the 1988-2007 era |
| **Long, off the highs** | Clearly net long, but less than usual for the last 3 years, with more shorts active |
| **Crowded long** | Heavily net long and near the top of the 3-year range: "everyone is already in" |

In [ ]:
gold_regimes = regimes.assign_regimes(gold_features)  # K-Means, 4 groups
gold_profile = regimes.regime_profile(gold_regimes)
gold_profile.set_index("name")[["weeks", "share_of_history", "typical_stay_weeks", "long_share", "short_share",
                                "net_share", "cot_index"]].rename(columns={
    "weeks": "Weeks", "share_of_history": "Share of history", "typical_stay_weeks": "Typical stay (weeks)",
    "long_share": "Longs (share)", "short_share": "Shorts (share)", "net_share": "Net (share)",
    "cot_index": "3-yr percentile"}).round(2)

**The map below:** every dot is one weekly report, placed by speculators' longs (across) and shorts (up).
Colour and marker shape show the regime; the big marker is the regime's average week. The diagonal line is
where longs equal shorts: dots above it are net short.

In [ ]:
rplots.regime_map_figure(gold_regimes, gold_profile);

**The fingerprint below:** the average week of each regime (the numbers), shaded red when higher than a
typical week and blue when lower. Read across a row to describe a regime; read down a column to compare
regimes on one measure.

In [ ]:
rplots.fingerprint_figure(gold_regimes, gold_profile);

### Step 4: regimes through time
**Top panel:** speculators' net position each week, coloured by regime. **Bottom strip:** which regime each
week was in. **Then a grid** of the chance of moving from this week's regime (row) to next week's (column).

In [ ]:
rplots.regime_timeline_figure(gold_regimes);

In [ ]:
rplots.transition_figure(regimes.transition_matrix(gold_regimes['regime']));

**What we found:**
- Net-short spells fill the 1990s and early 2000s; since then that regime appears only at 3-year lows
  (2013-2019). Crowded longs dominate since the mid-2000s.
- **Regimes are sticky:** an 82-90% chance of being in the same regime next week, so a regime typically lasts
  6-10 weeks.
- **Moves go to neighbours:** net short almost never jumps straight to crowded long (0%). Positioning shifts
  gradually.

### Step 5: does the regime tell us what speculators do next?
**The question, written as a test:**
- **H0 (nothing going on):** speculators' change over the next 8 weeks is the same whichever regime we start in.
- **Ha (something going on):** it differs by regime.

We use only every 8th week, so the 8-week windows don't overlap and inflate the evidence.

**The chart:** left, the spread of what speculators did next (the box holds the middle half of weeks, the
black line is the typical week, right of 0 = they bought). Right, how often they ended up more long.

In [ ]:
forward_test = regimes.forward_flow_test(gold_regimes)
print(f"p-value = {forward_test['p_value']:.6f} on {forward_test['weeks_tested']} non-overlapping weeks")
rplots.what_happens_next_figure(gold_regimes, gold_profile, forward_test);

**What we found:** the p-value is far below 0.05, so we **reject H0**: the regime does tell us what
speculators tend to do next.
- After **Crowded long** weeks, speculators were *less* long 8 weeks later 61% of the time: crowded bets
  tend to be trimmed.
- After **Net short / 3-yr low** weeks, they were *more* long 73% of the time: short bets tend to be bought
  back ("short covering").
- In other words, **positioning tends to swing back towards the middle.**

### Step 6: how confidently is each week assigned?
**The chart:** each thin sliver is one week; the longer it reaches right, the more clearly that week belongs to
its regime. Slivers left of 0 sit closer to another regime (border cases).

In [ ]:
rplots.silhouette_figure(state_matrix, gold_regimes['regime'].to_numpy());

**What we found:** almost every week is on the positive side, with an average of 0.37: most weeks belong
clearly to their regime, and only a handful are border cases.

## Part 4: What does the gold price say?

### Step 7a: the regimes on the price chart
**The chart:** gold's price since 2000 (log scale, so equal heights mean equal % moves), each week coloured by
its regime.

In [ ]:
rplots.price_timeline_figure(gold_regimes);

**What we see:** crowded longs come with strong rallies (2005-2011, 2019-2026); net-short or 3-year-low
weeks come with weak prices (2000-2001, 2013, 2015 and 2018). Positioning looks like a **mirror of the
trend**. The next two charts check whether it is a mirror (following the price) or a forecast (leading it).

### Step 7b: do speculators move before, with, or after the price?
**The chart:** each bar is the correlation between speculators' change in a week and gold's move in another
week. Orange bars: gold's week came first. Grey bar: the same week. Blue bars: gold's week came after. If
speculators could forecast gold, the blue bars would stand tall.

In [ ]:
lead_lag = price_data.lead_lag_correlations(gold_features)
rplots.lead_lag_figure(lead_lag);

**What we found:**
- **Same week: 0.51.** Weeks when gold rises are clearly weeks when speculators buy, and vice versa.
- **Previous week: 0.12.** They also partly react to *last* week's move: catching up with the trend.
- **Following weeks: all close to 0.** Their buying or selling says nothing about gold's next moves.

So **speculators move with the price, not ahead of it.**

### Step 7c: does the regime tell us where gold goes next?
**The same test as Step 5, but for gold's price:**
- **H0:** gold's return over the next 8 weeks is the same whichever regime we start in.
- **Ha:** it differs by regime.

To avoid cheating, the 8 weeks start from the first close **after the report is published** (Friday afternoon),
not from the Tuesday the positions refer to.

In [ ]:
price_test = regimes.forward_flow_test(gold_regimes, column=regimes.FORWARD_RETURN_COLUMN)
print(f"p-value = {price_test['p_value']:.2f} on {price_test['weeks_tested']} non-overlapping weeks")
display(gold_profile.set_index("name")[["forward_return", "share_price_rose", "weeks_with_price"]].rename(
    columns={"forward_return": "Gold's avg. return next 8 wks", "share_price_rose": "Share of weeks gold rose",
             "weeks_with_price": "Weeks with a price"}).round(3))
rplots.gold_after_regime_figure(gold_regimes, gold_profile, price_test);

**What we found:** the p-value is about 0.16, well above 0.05, so we **cannot reject H0**: the regimes
do **not** forecast gold's price.
- Gold rose about 1.8% per 8 weeks on average over this period (a long bull market), and its average return
  was positive after every regime.
- "Low participation" looks best (gold up 83% of the time), but that is only 70 weeks, almost all in the
  2001-2007 bull market: an era effect, not a signal.
- The classic idea that "crowded longs mean the top is in" is **not** supported for gold's price here: after
  crowded-long weeks gold still rose 59% of the time.

### Step 7d: re-running the clustering with the price included
We repeat the K-Means grouping three ways on the same weeks (2001 onwards):
1. **Positioning only** (what we have used so far),
2. **Positioning + gold price** (adding where gold sits in its 3-year range and its last-13-week trend),
3. **Gold price only.**

For each, we score separation and test whether the groups forecast what speculators do next and what gold does
next (p-values below the 0.05 line = yes). *This cell takes about 15 seconds.*

In [ ]:
feature_sets = {
    "Positioning only (kept)": config.STATE_FEATURES,
    "Positioning + gold price": config.STATE_FEATURES + config.PRICE_FEATURES,
    "Gold price only": config.PRICE_FEATURES,
}
comparison = regimes.feature_set_comparison(gold_features, feature_sets)
display(comparison[comparison["k"] == 4].set_index("features")[
    ["silhouette", "stability", "p_positioning_next", "p_gold_next"]].rename(columns={
    "silhouette": "Separation", "stability": "Stability", "p_positioning_next": "p-value: speculators next",
    "p_gold_next": "p-value: gold next"}).round(3))
rplots.feature_set_figure(comparison);

**What we found:**
- **Adding the price makes the positioning groups blurrier** (separation drops from about 0.32 to 0.25) because
  price and positioning largely carry the same information: they move together.
- **Price-only groups are cleaner**, but they just sort price trends (rising vs falling markets).
- **No version forecasts gold's next 8 weeks** (every p-value in panel 3 is above 0.05).
- So we **keep the positioning-only regimes**, and use the price as a yardstick to check them rather than as an
  input.

### Step 8: regimes vs your weekly signal
Your weekly signal (Bullish, Bearish, ...) describes **this week's change**; the regime describes **where
positioning stands**. The table shows, for each regime, how often each weekly signal appeared (rows add up
to 1).

In [ ]:
weekly_regime = gold_regimes[["date", "regime_name"]].assign(date=lambda d: d["date"].dt.strftime("%Y-%m-%d"))
gold_regime_df = gold_df[gold_df["market"] == config.REGIME_MARKET].merge(weekly_regime, on="date", how="inner")
pd.crosstab(gold_regime_df["regime_name"], gold_regime_df["Interpretation"], normalize="index").round(2)

**What we found:** in crowded-long weeks the weekly signal is most often *Bullish* (speculators still
adding), and in net-short weeks most often *Bearish*. The weekly signal follows the crowd; the regime points to
the swing back that tends to follow **in positioning** (not in price).

## Summary: what this all means, in plain words

1. **Gold speculators move between four recurring positioning states** (regimes): net short or at a 3-year
   low, low participation (mostly before 2007), long but off the highs, and crowded long. K-Means found them
   without being told what to look for, and they come back reliably when the data is resampled.
2. **Regimes are sticky** (they typically last 6-10 weeks) **and change gradually**.
3. **Positioning swings back towards the middle:** crowded longs tend to be trimmed and net shorts tend to be
   bought back over the following 8 weeks. This is statistically solid.
4. **Speculators move with the gold price, not ahead of it:** they buy in the weeks gold rises and sell in the
   weeks it falls, and their positioning tells us nothing about gold's next weeks.
5. **The regimes do not forecast gold's price,** and adding the price to the clustering did not change that.
   COT positioning is best read as a **description of the crowd**, a mirror of the trend so far, not a
   price-timing signal on its own.
6. The latest report (2026-10-13) is in **Long, off the highs**: still heavily net long, but below the top of
   its 3-year range.

**Caveats:** the model describes the past (it was fitted on all of history, not tested on unseen future data);
the price analysis covers 2000-2026 only, which was mostly a gold bull market; nothing here is a trading
recommendation.

**Possible next steps:** run the same analysis for silver, the euro or other markets; check shorter horizons
(1-2 weeks) or the size of moves rather than direction; try a time-aware model (a hidden Markov model) that
learns how regimes hand over to each other.

The same results, with all figures, are written up in `reports/gold_regimes.md`
(`python scripts/gold_regimes.py` regenerates it).

## Further reading

Beginner-friendly first:
1. **StatQuest with Josh Starmer** (short, friendly YouTube videos). Search "StatQuest K-means", "StatQuest
   PCA", "StatQuest DBSCAN", "StatQuest p-values": https://www.youtube.com/@statquest
2. James, Witten, Hastie & Tibshirani, ***An Introduction to Statistical Learning*** (2nd ed., free PDF),
   chapter 12 "Unsupervised Learning": https://www.statlearning.com
3. scikit-learn user guide, **"Clustering"**: pictures of every method on example data:
   https://scikit-learn.org/stable/modules/clustering.html
4. scikit-learn example, **"Selecting the number of clusters with silhouette analysis"**:
   https://scikit-learn.org/stable/auto_examples/cluster/plot_kmeans_silhouette_analysis.html
5. CFTC, **"Commitments of Traders: explanatory notes"** (what each COT column means):
   https://www.cftc.gov/MarketReports/CommitmentsofTraders/ExplanatoryNotes/index.htm
6. Briese, S. (2008). ***The Commitments of Traders Bible***. Wiley. (How traders use COT data, including the
   3-year "COT index".)
7. Wang, C. (2003). The behavior and performance of major types of futures traders. *Journal of Futures
   Markets*, 23(1), 1-31.

Original papers behind the methods (more technical):
8. Lloyd, S. (1982). Least squares quantization in PCM. *IEEE Transactions on Information Theory*, 28(2),
   129-137. (K-Means)
9. Ester, M., Kriegel, H.-P., Sander, J. & Xu, X. (1996). A density-based algorithm for discovering clusters
   in large spatial databases with noise. *Proceedings of KDD-96*, 226-231. (DBSCAN)
10. Rousseeuw, P. J. (1987). Silhouettes: a graphical aid to the interpretation and validation of cluster
    analysis. *Journal of Computational and Applied Mathematics*, 20, 53-65.
11. Hubert, L. & Arabie, P. (1985). Comparing partitions. *Journal of Classification*, 2, 193-218. (Stability
    score)
12. Schwarz, G. (1978). Estimating the dimension of a model. *Annals of Statistics*, 6(2), 461-464. (BIC)
13. Kruskal, W. H. & Wallis, W. A. (1952). Use of ranks in one-criterion variance analysis. *Journal of the
    American Statistical Association*, 47(260), 583-621.